In [22]:
import torch
import torch.nn as nn
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset,DataLoader
from torchvision.models import vgg19,VGG19_Weights
import torch.optim as optim
import torchvision.transforms as T

In [23]:
device = torch.device('mps' if torch.mps.is_available else 'cpu')
model = vgg19(weights=VGG19_Weights.DEFAULT).to(device)

In [24]:
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225])
])
train = CIFAR10(root='./data',train=True,transform=transform)
test = CIFAR10(root='./data',train=False,transform=transform)

train_loader = DataLoader(dataset=train,batch_size=32,shuffle=True)
test_loader = DataLoader(dataset=test,batch_size=32,shuffle=False)

In [25]:
num_features = model.classifier[6].in_features
out_features = len(test.classes)
model.classifier[6] = nn.Linear(in_features=num_features,out_features=out_features)

In [26]:
for name, param in model.named_parameters():
    if not name.startswith('classifier'):
        param.requires_grad = False

In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr=1e-3)

In [28]:
from tqdm import tqdm
def train_loop(model,train_loader,criterion,optimizer):
    model.train()
    run_loss, correct ,total = 0,0,0
    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits,labels)
        loss.backward()
        optimizer.step()
        preds = torch.argmax(logits,dim=1)
        run_loss += loss.item()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = run_loss/len(train_loader)
    epoch_acc = correct/total
    print(f'Epoch Loss : {epoch_loss:.4f} | Epoch Acc : {epoch_acc:.4f}')

In [19]:
for epoch in range(1):
    print(f'Epoch : {epoch+1}')
    train_loop(model=model,train_loader=train_loader,criterion=criterion,optimizer=optimizer)

Epoch : 1


  0%|                                         | 1/1563 [00:01<41:39,  1.60s/it]

32


  0%|                                         | 2/1563 [00:03<40:31,  1.56s/it]

32


...

32


  0%|▏                                        | 6/1563 [00:09<40:01,  1.54s/it]

32


  0%|▏                                        | 6/1563 [00:10<46:50,  1.81s/it]


KeyboardInterrupt: 

In [21]:
@torch.no_grad()
def test_loop(model,test_loader,criterion,optimizer):
    model.eval()
    correct, total = 0,0
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        preds = torch.argmax(logits,dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    print(f"Accuracy : {acc:.4f}")